# Dataset annotation agreement

Measure agreement between the three independent annotators.

Method and inputs: [workflow guide](../../docs/workflow.md).

In [1]:
import sys
from pathlib import Path

for _parent in (Path.cwd(), *Path.cwd().parents):
    if (_parent / "notebooks" / "_shared.py").is_file():
        sys.path.insert(0, str(_parent / "notebooks"))
        break
else:
    raise FileNotFoundError("Start Jupyter inside the repository.")

from _shared import (
    project_root, data_directory, results_directory, load_environment,
    require_environment, cache_directory, cache_fingerprint,
    load_cached_index, save_cached_index, metadata_directory, pdf_directory,
    reference_datasets, review_records, case_study_filters, align_full_text_results,
)

load_environment()

import glob
import numpy as np
import os
import pandas as pd

from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa

folder_path = data_directory()
result_path = results_directory("gemini")
annotation_path = str(Path(data_directory()) / "annotations") + os.sep

In [2]:
CATEGORIES = ['AQA-ready', 'AQA-Related', 'Not Applicable to AQA']

def parse_annotator_file(filepath, rater_name):
    df = pd.read_excel(filepath, skiprows=2)
    df = df.rename(columns={df.columns[0]: 'Dataset Name'})
    df_labels = df.iloc[:, 3:6]

    def get_label(row):
        row_vals = row.fillna(0).values
        if row_vals.max() > 0:
            return CATEGORIES[np.argmax(row_vals)]
        return np.nan

    df[rater_name] = df_labels.apply(get_label, axis=1)

    return df[['Dataset Name', rater_name]].dropna(subset=['Dataset Name'])

In [3]:
file_pattern = os.path.join(annotation_path, 'annotator_*.xlsx')
annotator_files = glob.glob(file_pattern)

master_df = None

for i, filepath in enumerate(annotator_files):
    rater_name = f'Rater_Ext_{i+1}'
    rater_df = parse_annotator_file(filepath, rater_name)

    if master_df is None:
        master_df = rater_df
    else:
        master_df = pd.merge(master_df, rater_df, on='Dataset Name', how='outer')

In [4]:
print("--- Summary Table (Master Table) ---")
print(master_df)
print("------------------------------------")

rater_columns = master_df.columns[1:]
complete_ratings_df = master_df.dropna(subset=rater_columns).copy()
excluded_datasets = len(master_df) - len(complete_ratings_df)
ratings_matrix = complete_ratings_df[rater_columns].astype(str).values

agg_data, categories_found = aggregate_raters(ratings_matrix)

# Fleiss' Kappa
kappa = fleiss_kappa(agg_data, method='fleiss')

print(f"\nInter-Rater Reliability Results:")
print(f"Number of analyzed datasets: {len(complete_ratings_df)}")
print(f"Datasets excluded because of missing ratings: {excluded_datasets}")
print(f"Number of raters: {len(rater_columns)}")
print(f"Categories detected: {categories_found}")
print(f"Fleiss' Kappa: {kappa:.4f}")

if kappa > 0.8:
    print("Agreement: Almost Perfect (Excellent)")
elif kappa > 0.6:
    print("Agreement: Substantial")
elif kappa > 0.4:
    print("Agreement: Moderate")
else:
    print("Agreement: Fair/Poor")

print("\n--- Disagreements Overview ---")
disagreements_df = complete_ratings_df[complete_ratings_df[rater_columns].nunique(axis=1) > 1]

if disagreements_df.empty:
    print("No disagreements found! All raters are in perfect agreement.")
else:
    print(f"Found {len(disagreements_df)} datasets with disagreements:\n")
    print(disagreements_df.to_string(index=False))
print("------------------------------")

--- Summary Table (Master Table) ---
                   Dataset Name  Rater_Ext_1  Rater_Ext_2  Rater_Ext_3
0                  3D Animation    AQA-ready    AQA-ready    AQA-ready
1                  AGF-Olympics    AQA-ready    AQA-ready    AQA-ready
2                ActivityNet1.3  AQA-Related  AQA-Related  AQA-Related
3                        CAD-60  AQA-Related  AQA-Related  AQA-Related
4                     CPR-Coach    AQA-ready    AQA-ready    AQA-ready
5                       Countix  AQA-Related  AQA-Related  AQA-Related
6                      Diving48  AQA-Related  AQA-Related  AQA-Related
7                        ExPose    AQA-ready    AQA-ready    AQA-ready
8                        FINA09    AQA-ready    AQA-ready    AQA-ready
9                        FS1000    AQA-ready    AQA-ready    AQA-ready
10                    FineRehab    AQA-ready    AQA-ready    AQA-ready
11                  Fitness-AQA    AQA-ready    AQA-ready    AQA-ready
12            Golf Swing Action  AQA-Rel